# Governed MCP — Paper Figures

Generates all figures and tables for the paper from experimental results.
Data sources: `results/results.db` (SQLite) and `analysis/tcr_results.csv`.

Figures:
- **Figure 1**: TSA vs N curve (4 models × 2 conditions)
- **Figure 2**: TCR vs N + secondary axis for K
- **Figure 3**: Latency CDF (P50/P99) + stacked bar decomposition
- **Figure 4**: Policy compliance — UIR + ESR across 4 adversarial categories

Tables:
- **Table 1**: TSA at N=500 by model + Delta + McNemar p-value
- **Table 2**: TSA by task difficulty
- **Ablation chart**: 4-bar at N=500

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# ── Style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'figure.dpi': 150,
})
COLORS = {
    'governed': '#1f77b4',
    'baseline': '#d62728',
    'random_k': '#ff7f0e',
    'domain': '#2ca02c',
    'full_attr': '#1f77b4',
}
MODEL_MARKERS = {'gpt-4o': 'o', 'gpt-4o-mini': 's',
                 'claude-3-5-sonnet-20241022': '^', 'meta-llama/Llama-3.1-70B-Instruct': 'D'}
MODEL_LABELS = {
    'gpt-4o': 'GPT-4o',
    'gpt-4o-mini': 'GPT-4o-mini',
    'claude-3-5-sonnet-20241022': 'Claude 3.5 Sonnet',
    'meta-llama/Llama-3.1-70B-Instruct': 'Llama 3.1 70B',
}
FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)

DB_PATH = 'results/results.db'

def load_db():
    conn = sqlite3.connect(DB_PATH)
    return conn

print('Style configured. Figures will be saved to:', FIGURES_DIR)

## Figure 1: TSA vs N (Tool Selection Accuracy)

In [ ]:
conn = load_db()
df = pd.read_sql_query("""
    SELECT r.model, tr.registry_size AS N, r.condition,
           AVG(tr.is_correct) AS TSA,
           COUNT(tr.is_correct) AS n_tasks,
           SQRT(AVG(tr.is_correct) * (1 - AVG(tr.is_correct)) / COUNT(tr.is_correct)) * 1.96 AS ci95
    FROM task_results tr
    JOIN runs r ON r.run_id = tr.run_id
    WHERE r.mode = 'tsa' AND r.condition IN ('baseline', 'governed')
    GROUP BY r.model, tr.registry_size, r.condition
""", conn)
conn.close()

if df.empty:
    print("No TSA data yet. Run experiments first.")
    # Demo data for figure layout development
    import itertools
    models = ['gpt-4o', 'gpt-4o-mini', 'claude-3-5-sonnet-20241022', 'meta-llama/Llama-3.1-70B-Instruct']
    N_levels = [10, 25, 50, 100, 200, 500]
    rows = []
    for model, n, cond in itertools.product(models, N_levels, ['baseline', 'governed']):
        base_tsa = 0.95 - 0.27 * np.log10(n / 10 + 1)
        governed_tsa = 0.94 - 0.02 * np.log10(n / 10 + 1)
        tsa = governed_tsa if cond == 'governed' else base_tsa
        rows.append({'model': model, 'N': n, 'condition': cond, 'TSA': tsa, 'ci95': 0.02})
    df = pd.DataFrame(rows)

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
axes = axes.flatten()

for ax, model in zip(axes, list(MODEL_LABELS.keys())):
    model_df = df[df['model'] == model]
    for condition, ls in [('governed', '-'), ('baseline', '--')]:
        cond_df = model_df[model_df['condition'] == condition].sort_values('N')
        if cond_df.empty:
            continue
        label = 'Governed' if condition == 'governed' else 'Baseline'
        color = COLORS[condition]
        ax.plot(cond_df['N'], cond_df['TSA'] * 100, ls=ls, color=color,
                marker=MODEL_MARKERS.get(model, 'o'), markersize=5, label=label, linewidth=1.5)
        ax.fill_between(cond_df['N'],
                         (cond_df['TSA'] - cond_df['ci95']) * 100,
                         (cond_df['TSA'] + cond_df['ci95']) * 100,
                         alpha=0.15, color=color)

    ax.set_xscale('log')
    ax.set_xlabel('Registry Size (N)')
    ax.set_ylabel('TSA (%)')
    ax.set_title(MODEL_LABELS.get(model, model))
    ax.set_xticks([10, 25, 50, 100, 200, 500])
    ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
    ax.set_ylim(50, 102)
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle('Figure 1: Tool Selection Accuracy vs Registry Size', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig1_tsa_vs_n.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig1_tsa_vs_n.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figure 1 saved.')

## Figure 2: Token Compression Ratio (TCR)

In [ ]:
tcr_path = Path('analysis/tcr_results.csv')
if tcr_path.exists():
    tcr_df = pd.read_csv(tcr_path)
else:
    print("TCR data not found. Run: python analysis/tcr.py --mode tcr")
    # Demo data
    tcr_df = pd.DataFrame({
        'N': [10, 25, 50, 100, 200, 500],
        'K': [8, 18, 35, 65, 120, 40],
        'TCR': [0.15, 0.28, 0.42, 0.58, 0.74, 0.92],
        'compression_pct': [15, 28, 42, 58, 74, 92],
    })

fig, ax1 = plt.subplots(figsize=(7, 4))
ax2 = ax1.twinx()

# Primary: TCR
ax1.plot(tcr_df['N'], tcr_df['compression_pct'], 'o-', color='#1f77b4',
         linewidth=2, markersize=6, label='TCR (%)')
ax1.fill_between(tcr_df['N'], 0, tcr_df['compression_pct'], alpha=0.1, color='#1f77b4')
ax1.set_xscale('log')
ax1.set_xlabel('Registry Size (N)')
ax1.set_ylabel('Token Compression Ratio (%)', color='#1f77b4')
ax1.tick_params(axis='y', labelcolor='#1f77b4')
ax1.set_xticks([10, 25, 50, 100, 200, 500])
ax1.xaxis.set_major_formatter(mticker.ScalarFormatter())
ax1.set_ylim(0, 100)
ax1.grid(True, alpha=0.3)

# Secondary: K (tools returned after filtering)
ax2.bar(range(len(tcr_df)), tcr_df['K'], alpha=0.3, color='#ff7f0e', width=0.4, label='K (filtered tools)')
ax2.set_ylabel('K — Tools After Filtering', color='#ff7f0e')
ax2.tick_params(axis='y', labelcolor='#ff7f0e')
ax2.set_xticks(range(len(tcr_df)))
ax2.set_xticklabels(tcr_df['N'].tolist())

# Annotate N=500
idx_500 = tcr_df[tcr_df['N'] == 500].index
if len(idx_500) > 0:
    i = idx_500[0]
    tcr_val = tcr_df.loc[i, 'compression_pct']
    k_val = tcr_df.loc[i, 'K']
    ax1.annotate(f'TCR={tcr_val:.0f}%\nK={k_val}',
                 xy=(500, tcr_val), xytext=(350, tcr_val - 15),
                 arrowprops=dict(arrowstyle='->', color='gray'),
                 fontsize=9)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title('Figure 2: Token Compression Ratio vs Registry Size', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig2_tcr.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig2_tcr.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figure 2 saved.')

## Figure 3: Latency CDF + Stacked Bar Decomposition

In [ ]:
timing_path = Path('results/latency/timing_breakdown.csv')

if timing_path.exists():
    timing_df = pd.read_csv(timing_path)
else:
    print("Latency data not found. Run Locust load test first.")
    # Demo data (realistic synthetic latency)
    rng = np.random.default_rng(42)
    n = 5000
    timing_df = pd.DataFrame({
        'jwt_ms':    rng.lognormal(0.3, 0.3, n),
        'query_ms':  rng.lognormal(1.5, 0.5, n),
        'abac_ms':   rng.lognormal(-0.5, 0.3, n),
        'serial_ms': rng.lognormal(0.0, 0.3, n),
    })
    timing_df['proxy_total_ms'] = timing_df[['jwt_ms','query_ms','abac_ms','serial_ms']].sum(axis=1)
    timing_df['total_ms'] = timing_df['proxy_total_ms'] + rng.lognormal(0.5, 0.4, n)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# ── CDF ──────────────────────────────────────────────────────────────────
latencies = np.sort(timing_df['total_ms'].values)
cdf = np.arange(1, len(latencies) + 1) / len(latencies)
ax1.plot(latencies, cdf * 100, color='#1f77b4', linewidth=1.5)

# Annotate P50 and P99
p50 = np.percentile(latencies, 50)
p99 = np.percentile(latencies, 99)
ax1.axvline(p50, color='#2ca02c', linestyle='--', linewidth=1, label=f'P50={p50:.1f}ms')
ax1.axvline(p99, color='#d62728', linestyle='--', linewidth=1, label=f'P99={p99:.1f}ms')
ax1.axhline(50, color='gray', linestyle=':', alpha=0.5)
ax1.axhline(99, color='gray', linestyle=':', alpha=0.5)
ax1.set_xlabel('Latency (ms)')
ax1.set_ylabel('Percentile (%)')
ax1.set_title('(a) Request Latency CDF')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, p99 * 1.3)

# ── Stacked bar decomposition ─────────────────────────────────────────────
stages = ['jwt_ms', 'query_ms', 'abac_ms', 'serial_ms']
labels = ['JWT Verify', 'MongoDB Query', 'ABAC Eval', 'Serialization']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#9467bd']

# Compute P50 and P99 per stage
p50_stages = [np.percentile(timing_df[s], 50) for s in stages]
p99_stages = [np.percentile(timing_df[s], 99) for s in stages]

x = np.array([0, 1])
bar_width = 0.5
bottoms_50 = np.zeros(1)
bottoms_99 = np.zeros(1)

for val50, val99, label, color in zip(p50_stages, p99_stages, labels, colors):
    ax2.bar([0], val50, bar_width, bottom=bottoms_50, label=label, color=color, alpha=0.85)
    ax2.bar([1], val99, bar_width, bottom=bottoms_99, color=color, alpha=0.85)
    bottoms_50 += val50
    bottoms_99 += val99

ax2.set_xticks([0, 1])
ax2.set_xticklabels(['P50', 'P99'])
ax2.set_ylabel('Latency (ms)')
ax2.set_title('(b) Latency Decomposition')
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')

fig.suptitle('Figure 3: Proxy Discovery Latency (N=500, 100 Concurrent Agents)', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig3_latency.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig3_latency.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Figure 3 saved. P50={p50:.2f}ms  P99={p99:.2f}ms')

## Figure 4: Policy Compliance (Adversarial)

In [ ]:
adv_path = Path('analysis/adversarial_metrics.csv')

if adv_path.exists():
    adv_df = pd.read_csv(adv_path)
else:
    print("Adversarial metrics not found. Run: python analysis/tcr.py --mode adversarial")
    # Demo data
    adv_df = pd.DataFrame([
        {'model': 'gpt-4o', 'condition': 'governed',  'category': 'A', 'UIR': 0.42, 'PBR': 1.0, 'ESR': 1.0},
        {'model': 'gpt-4o', 'condition': 'governed',  'category': 'B', 'UIR': 0.30, 'PBR': 1.0, 'ESR': 1.0},
        {'model': 'gpt-4o', 'condition': 'governed',  'category': 'C', 'UIR': 0.28, 'PBR': 1.0, 'ESR': 1.0},
        {'model': 'gpt-4o', 'condition': 'governed',  'category': 'D', 'UIR': 0.55, 'PBR': 1.0, 'ESR': 1.0},
        {'model': 'gpt-4o', 'condition': 'llm_only',  'category': 'A', 'UIR': 0.42, 'PBR': 0.0, 'ESR': 0.58},
        {'model': 'gpt-4o', 'condition': 'llm_only',  'category': 'B', 'UIR': 0.30, 'PBR': 0.0, 'ESR': 0.70},
        {'model': 'gpt-4o', 'condition': 'llm_only',  'category': 'C', 'UIR': 0.28, 'PBR': 0.0, 'ESR': 0.72},
        {'model': 'gpt-4o', 'condition': 'llm_only',  'category': 'D', 'UIR': 0.55, 'PBR': 0.0, 'ESR': 0.45},
    ])

cat_labels = {'A': 'Direct\nInjection', 'B': 'Indirect\nInstruction',
               'C': 'Role\nEscalation', 'D': 'Multi-step\nDeception'}
categories = ['A', 'B', 'C', 'D']

# Use GPT-4o for primary figure; all models can be shown in appendix
model = adv_df['model'].iloc[0]  # Use first model
model_df = adv_df[adv_df['model'] == model]

gov = model_df[model_df['condition'] == 'governed'].set_index('category')
llm = model_df[model_df['condition'] == 'llm_only'].set_index('category')

x = np.arange(len(categories))
width = 0.2

fig, ax = plt.subplots(figsize=(9, 4.5))

# UIR bars
uir_gov = [gov.loc[c, 'UIR'] * 100 if c in gov.index else 0 for c in categories]
uir_llm = [llm.loc[c, 'UIR'] * 100 if c in llm.index else 0 for c in categories]
esr_gov = [gov.loc[c, 'ESR'] * 100 if c in gov.index else 0 for c in categories]
esr_llm = [llm.loc[c, 'ESR'] * 100 if c in llm.index else 0 for c in categories]

ax.bar(x - 1.5*width, uir_gov, width, label='UIR — Governed', color='#1f77b4', alpha=0.85)
ax.bar(x - 0.5*width, uir_llm, width, label='UIR — LLM Only', color='#d62728', alpha=0.85)
ax.bar(x + 0.5*width, esr_gov, width, label='ESR — Governed', color='#1f77b4', alpha=0.4, hatch='//')
ax.bar(x + 1.5*width, esr_llm, width, label='ESR — LLM Only', color='#d62728', alpha=0.4, hatch='//')

# PBR=100% annotation for governed
ax.axhline(100, color='#1f77b4', linestyle=':', linewidth=0.8, alpha=0.6)
ax.text(3.5, 101, 'PBR=100%', color='#1f77b4', fontsize=9, ha='right')

ax.set_xlabel('Adversarial Category')
ax.set_ylabel('Rate (%)')
ax.set_title(f'Figure 4: Policy Compliance — {MODEL_LABELS.get(model, model)}', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([cat_labels[c] for c in categories])
ax.set_ylim(0, 115)
ax.legend(ncol=2, fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig4_adversarial.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig4_adversarial.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figure 4 saved.')

## Table 1: TSA at N=500 by Model

In [ ]:
conn = load_db()
df500 = pd.read_sql_query("""
    SELECT r.model, r.condition,
           AVG(tr.is_correct) AS TSA,
           COUNT(*) AS n
    FROM task_results tr
    JOIN runs r ON r.run_id = tr.run_id
    WHERE r.mode = 'tsa' AND tr.registry_size = 500
      AND r.condition IN ('baseline', 'governed')
    GROUP BY r.model, r.condition
""", conn)

# McNemar results
mcnemar_df = pd.read_sql_query("""
    SELECT DISTINCT r.model FROM runs r WHERE r.mode = 'tsa'
""", conn) if not df500.empty else pd.DataFrame()
conn.close()

if df500.empty:
    print("No N=500 TSA data yet.")
else:
    table1 = df500.pivot(index='model', columns='condition', values='TSA').reset_index()
    table1['Delta'] = table1.get('governed', 0) - table1.get('baseline', 0)
    table1['model'] = table1['model'].map(lambda m: MODEL_LABELS.get(m, m))
    table1 = table1.sort_values('governed', ascending=False)

    print("\nTable 1: TSA at N=500 by Model")
    print(table1.to_string(index=False, float_format=lambda x: f"{x*100:.1f}%"))
    table1.to_csv('analysis/table1_tsa_n500.csv', index=False)
print()

## Table 2: TSA by Task Difficulty

In [ ]:
conn = load_db()
diff_df = pd.read_sql_query("""
    SELECT r.model, r.condition, tr.difficulty,
           AVG(tr.is_correct) AS TSA, COUNT(*) AS n
    FROM task_results tr
    JOIN runs r ON r.run_id = tr.run_id
    WHERE r.mode = 'tsa' AND tr.registry_size = 500
      AND r.condition IN ('baseline', 'governed')
    GROUP BY r.model, r.condition, tr.difficulty
""", conn)
conn.close()

if diff_df.empty:
    print("No difficulty breakdown data yet.")
else:
    print("\nTable 2: TSA by Difficulty (N=500)")
    pivot = diff_df.pivot_table(index=['model', 'difficulty'], columns='condition', values='TSA')
    print(pivot.to_string(float_format=lambda x: f"{x*100:.1f}%"))
    diff_df.to_csv('analysis/table2_tsa_by_difficulty.csv', index=False)

## Ablation Chart: N=500 Conditions

In [ ]:
conn = load_db()
ablation_df = pd.read_sql_query("""
    SELECT r.model, r.condition, AVG(tr.is_correct) AS TSA, COUNT(*) AS n
    FROM task_results tr
    JOIN runs r ON r.run_id = tr.run_id
    WHERE r.mode = 'tsa' AND tr.registry_size = 500
      AND r.condition IN ('baseline', 'random_k', 'domain', 'full_attr')
    GROUP BY r.model, r.condition
""", conn)
conn.close()

if ablation_df.empty:
    # Demo data
    ablation_df = pd.DataFrame({
        'model': ['gpt-4o'] * 4,
        'condition': ['baseline', 'random_k', 'domain', 'full_attr'],
        'TSA': [0.68, 0.75, 0.85, 0.95],
    })

ABLATION_LABELS = {'baseline': 'Full Registry', 'random_k': 'Random-K Subset',
                   'domain': 'Domain Filter', 'full_attr': 'Full Attribute\nFilter (Governed)'}
ABLATION_ORDER = ['baseline', 'random_k', 'domain', 'full_attr']
ABLATION_COLORS = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']

model = ablation_df['model'].iloc[0]
mdf = ablation_df[ablation_df['model'] == model]

fig, ax = plt.subplots(figsize=(7, 4))
x = range(len(ABLATION_ORDER))
tsas = [mdf[mdf['condition'] == c]['TSA'].values[0] * 100
        if c in mdf['condition'].values else 0
        for c in ABLATION_ORDER]

bars = ax.bar(x, tsas, color=ABLATION_COLORS, alpha=0.85, width=0.5)
for bar, val in zip(bars, tsas):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=10)

ax.set_xticks(list(x))
ax.set_xticklabels([ABLATION_LABELS[c] for c in ABLATION_ORDER], fontsize=9)
ax.set_ylabel('TSA (%)')
ax.set_ylim(50, 105)
ax.set_title(f'Ablation: Filtering Strategy at N=500 ({MODEL_LABELS.get(model, model)})',
             fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_ablation.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig_ablation.png', bbox_inches='tight', dpi=150)
plt.show()
print('Ablation chart saved.')

## Consistency Checker (T-38)

Verifies that numbers cited in the paper match the SQLite database.

In [ ]:
# Edit PAPER_CLAIMS to match the numbers you write in the paper.
# Run this cell to verify claims against the SQLite results.

PAPER_CLAIMS = [
    # (description, model, condition, N, claimed_TSA)
    # Example: ("GPT-4o Governed N=500", "gpt-4o", "governed", 500, 0.95),
]

if PAPER_CLAIMS:
    conn = load_db()
    for desc, model, condition, n, claimed in PAPER_CLAIMS:
        actual = pd.read_sql_query("""
            SELECT AVG(tr.is_correct) AS TSA FROM task_results tr
            JOIN runs r ON r.run_id = tr.run_id
            WHERE r.model = ? AND r.condition = ? AND tr.registry_size = ? AND r.mode = 'tsa'
        """, conn, params=(model, condition, n))['TSA'].values[0]

        diff = abs((actual or 0) - claimed)
        status = '✓' if diff < 0.005 else '✗'
        print(f"  {status} {desc}: claimed={claimed:.3f}  actual={actual:.3f}  diff={diff:.4f}")
    conn.close()
else:
    print("Add paper claims to PAPER_CLAIMS list above after writing the paper.")